# PySpark drills 3: types, functions and nulls

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/pyspark/pyspark-drills-03-types-functions-and-nulls.ipynb)

Practice for [PySpark 3: types, functions and nulls](pyspark-03-types-functions-and-nulls.ipynb). Each exercise uses the `customers` and `orders` tables, or small inputs created in its data cell.

Write your answer in the empty cell under each exercise, run it, and compare your output with the expected output.
The reference answer is hidden under the exercise; open it when you are done or stuck.

*Last reviewed: November 2026, against Apache Spark 4.0.*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0

## Setup and sample data

The first cell starts a local Spark session. The second creates `customers` (one row per customer; customer 4 has no region, and 7 and 8 have never ordered) and `orders` (one row per order; order 12 lands just after midnight UTC): a slice of the orders platform that every
module shares, introduced in [PySpark 1](pyspark-01-how-spark-runs.ipynb#setup-and-sample-data).

In [ ]:
import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable  # Python workers use this notebook's interpreter
os.environ["TZ"] = "UTC"  # the sample timestamps are UTC
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-")  # scratch space for the files the lessons write

spark = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")  # the default of 200 is far too many for tiny data
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
from datetime import date, datetime
from decimal import Decimal as D

from pyspark.sql import functions as F

customers = spark.createDataFrame([
    (1, "Acme Corp", "North", date(2024, 1, 3)),
    (2, "Bright Retail", "North", date(2024, 2, 11)),
    (3, "Cyan Traders", "South", date(2024, 3, 15)),
    (4, "Delta Grocers", None, date(2024, 5, 2)),  # region never arrived from the CRM
    (5, "Echo Supplies", "West", date(2024, 6, 20)),
    (6, "Foxtrot Foods", "West", date(2024, 8, 1)),
    (7, "Gamma Sports", "East", date(2024, 9, 14)),  # 7 and 8 have never ordered
    (8, "Hotel Homeware", "East", date(2024, 11, 5)),
], "customer_id int, name string, region string, signup_date date")

orders = spark.createDataFrame([
    (1, 1, "paid", datetime(2026, 1, 5, 9, 15), D("120.00")),
    (2, 1, "shipped", datetime(2026, 1, 5, 10, 5), D("84.50")),
    (3, 1, "returned", datetime(2026, 1, 5, 11, 40), D("30.00")),
    (4, 2, "paid", datetime(2026, 1, 6, 15, 20), D("260.00")),
    (5, 2, "placed", datetime(2026, 1, 7, 8, 5), D("55.25")),
    (6, 3, "shipped", datetime(2026, 1, 7, 17, 45), D("149.00")),
    (7, 3, "paid", datetime(2026, 1, 9, 12, 10), D("310.00")),
    (8, 4, "returned", datetime(2026, 1, 10, 19, 30), D("89.90")),
    (9, 4, "shipped", datetime(2026, 1, 12, 8, 55), D("1180.00")),
    (10, 5, "paid", datetime(2026, 1, 12, 13, 0), D("74.00")),
    (11, 6, "placed", datetime(2026, 1, 13, 10, 25), D("210.00")),
    (12, 6, "shipped", datetime(2026, 1, 14, 0, 30), D("96.40")),  # just after midnight UTC
], "order_id int, customer_id int, status string, order_ts timestamp, amount decimal(12,2)")

## Exercise 1: Type raw amounts and flag rejects

**Difficulty:** Easy · about 10 min

Convert `amount_text` to `decimal(12,2)` without failing on bad values. Return `order_id`, `amount` and
`is_rejected`, which is true when a value was delivered but could not be converted. Order by `order_id`.

**Sample input**

**`raw`**

| order_id | amount_text |
| --- | --- |
| o1 | 120.50 |
| o2 | N/A |
| o3 | null |
| o4 | 7.5 |

**Expected output**

| order_id | amount | is_rejected |
| --- | --- | --- |
| o1 | 120.50 | false |
| o2 | null | true |
| o3 | null | false |
| o4 | 7.50 | false |

In [ ]:
# Sample data for this exercise.
raw = spark.createDataFrame([("o1", "120.50"), ("o2", "N/A"), ("o3", None), ("o4", "7.5")], "order_id string, amount_text string")

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
result = (
    raw.withColumn("amount", F.expr("try_cast(amount_text AS DECIMAL(12,2))"))
    .withColumn("is_rejected", F.col("amount_text").isNotNull() & F.col("amount").isNull())
    .select("order_id", "amount", "is_rejected")
    .orderBy("order_id")
)
result.show()

## Exercise 2: Clean emails and extract the domain

**Difficulty:** Medium · about 10 min

Trim and lowercase each email. A null or blank email becomes `unknown`, and its domain is null. Return `id`,
`email_clean` and `domain`, ordered by `id`.

**Sample input**

**`contacts`**

| id | email |
| --- | --- |
| 1 |   Ada@Example.COM  |
| 2 |  |
| 3 | null |
| 4 | grace@Example.org |

**Expected output**

| id | email_clean | domain |
| --- | --- | --- |
| 1 | ada@example.com | example.com |
| 2 | unknown | null |
| 3 | unknown | null |
| 4 | grace@example.org | example.org |

In [ ]:
# Sample data for this exercise.
contacts = spark.createDataFrame([(1, "  Ada@Example.COM "), (2, ""), (3, None), (4, "grace@Example.org")], "id int, email string")

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
cleaned = contacts.withColumn("email_clean", F.coalesce(F.nullif(F.lower(F.trim("email")), F.lit("")), F.lit("unknown")))
result = (
    cleaned.withColumn(
        "domain", F.when(F.col("email_clean").contains("@"), F.element_at(F.split("email_clean", "@"), -1))
    )
    .select("id", "email_clean", "domain")
    .orderBy("id")
)
result.show()

## Exercise 3: Orders per business date in Chicago

**Difficulty:** Medium · about 10 min

Order timestamps are in UTC. Count orders per business date in `America/Chicago` time. Return `chicago_date` and
`orders`, ordered by date.

**Expected output**

| chicago_date | orders |
| --- | --- |
| 2026-01-05 | 3 |
| 2026-01-06 | 1 |
| 2026-01-07 | 2 |
| 2026-01-09 | 1 |
| 2026-01-10 | 1 |
| 2026-01-12 | 2 |
| 2026-01-13 | 2 |

In [ ]:
# This exercise uses the sample tables created above.

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
result = (
    orders.groupBy(F.to_date(F.from_utc_timestamp("order_ts", "America/Chicago")).alias("chicago_date"))
    .agg(F.count("*").alias("orders"))
    .orderBy("chicago_date")
)
result.show()

## Exercise 4: Size tiers

**Difficulty:** Easy · about 5 min

Assign each order a tier: `large` for 500 or more, `medium` for 100 or more, otherwise `small`. Return `order_id`,
`amount` and `tier`, ordered by `order_id`.

**Expected output**

| order_id | amount | tier |
| --- | --- | --- |
| 1 | 120.00 | medium |
| 2 | 84.50 | small |
| 3 | 30.00 | small |
| 4 | 260.00 | medium |
| 5 | 55.25 | small |
| 6 | 149.00 | medium |
| 7 | 310.00 | medium |
| 8 | 89.90 | small |
| 9 | 1180.00 | large |
| 10 | 74.00 | small |
| 11 | 210.00 | medium |
| 12 | 96.40 | small |

In [ ]:
# This exercise uses the sample tables created above.

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
result = orders.select(
    "order_id", "amount",
    F.when(F.col("amount") >= 500, "large").when(F.col("amount") >= 100, "medium").otherwise("small").alias("tier"),
).orderBy("order_id")
result.show()

## Exercise 5: Customers per region with a default

**Difficulty:** Medium · about 10 min

Count customers per region, reporting a missing region as `unknown`. Return `region` and `customers`, ordered by
`region`.

**Expected output**

| region | customers |
| --- | --- |
| East | 2 |
| North | 2 |
| South | 1 |
| West | 2 |
| unknown | 1 |

In [ ]:
# This exercise uses the sample tables created above.

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
result = (
    customers.groupBy(F.coalesce("region", F.lit("unknown")).alias("region"))
    .agg(F.count("*").alias("customers"))
    .orderBy("region")
)
result.show()

**Next:** [PySpark 4: reading and writing data](pyspark-04-reading-and-writing-data.ipynb)